In [1]:
# Cell 1 — Imports and dynamic project paths

from pathlib import Path
import json, random, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from IPython.display import display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

search_locations = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
PROJECT_ROOT = next(
    (path for path in search_locations if (path / "data").exists() and (path / "02_chocolate_dataset_creation.ipynb").exists()),
    None
)
assert PROJECT_ROOT is not None, "Could not locate the Chocolathon repository root."

BUNDLE_DIR = PROJECT_ROOT / "data" / "chocolates" / "frozen" / "chocolate_dataset_v1"
METADATA_DIR = BUNDLE_DIR / "metadata"
EXPERIMENT_DIR = PROJECT_ROOT / "data" / "experiments" / "flavor_embeddings_v1"
SPLIT_DIR = EXPERIMENT_DIR / "splits"
OUTPUT_DIR = EXPERIMENT_DIR / "outputs"

SPLIT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Frozen dataset:", BUNDLE_DIR)
print("Experiment directory:", EXPERIMENT_DIR)
print("Random seed:", SEED)

Project root: /homes/mxqasim/Chocolathon
Frozen dataset: /homes/mxqasim/Chocolathon/data/chocolates/frozen/chocolate_dataset_v1
Experiment directory: /homes/mxqasim/Chocolathon/data/experiments/flavor_embeddings_v1
Random seed: 42


In [2]:
# Cell 2 — Load and validate the accepted frozen dataset

manifest_path = METADATA_DIR / "accepted_dataset_manifest.csv"
dataset = pd.read_csv(manifest_path)

required_columns = {"sample_id", "flavor_id", "flavor_name", "source_file", "bundle_relative_path"}
missing_columns = required_columns - set(dataset.columns)

assert manifest_path.exists(), f"Manifest not found: {manifest_path}"
assert not missing_columns, f"Missing manifest columns: {sorted(missing_columns)}"
assert dataset["sample_id"].is_unique, "Duplicate sample IDs detected."
assert dataset["review_decision"].eq("accept").all(), "Non-accepted samples found."

dataset["image_path"] = dataset["bundle_relative_path"].map(lambda path: BUNDLE_DIR / path)
dataset["image_exists"] = dataset["image_path"].map(Path.exists)

class_summary = (
    dataset.groupby(["flavor_id", "flavor_name"], as_index=False)
    .agg(samples=("sample_id", "count"), source_trays=("source_file", "nunique"))
    .sort_values(["samples", "flavor_name"])
    .reset_index(drop=True)
)

print("=" * 72)
print("FROZEN FLAVOR DATASET")
print("=" * 72)
print("Samples:", len(dataset))
print("Classes:", dataset["flavor_id"].nunique())
print("Source trays:", dataset["source_file"].nunique())
print("Missing images:", int((~dataset["image_exists"]).sum()))
print("Duplicate sample IDs:", int(dataset["sample_id"].duplicated().sum()))
print("Classes with multiple trays:", int(class_summary["source_trays"].gt(1).sum()))

assert len(dataset) == 889
assert dataset["flavor_id"].nunique() == 33
assert dataset["image_exists"].all()

display(class_summary)
print("\nPASS: frozen dataset is ready for experiment design")

FROZEN FLAVOR DATASET
Samples: 889
Classes: 33
Source trays: 37
Missing images: 0
Duplicate sample IDs: 0
Classes with multiple trays: 4


,flavor_id,flavor_name,samples,source_trays
0,cocoashot_amaretto,Amaretto,12,1
1,cocoashot_chocolate_cake,Chocolate Cake,16,1
2,truffle_grey_salt_caramel,Grey Salt Caramel,16,1
3,cocoashot_whiskey,Whiskey,16,1
4,truffle_pineapple_moscato,Pineapple & Moscato,18,1
5,truffle_lemon,Lemon,20,1
6,truffle_espresso_martini,Espresso Martini,22,1
7,truffle_strawberry_shortcake,Strawberry Shortcake,23,1
8,truffle_brownie_batter,Brownie Batter,24,1
9,truffle_caramel_apple_cider,Caramel Apple Cider,24,1



PASS: frozen dataset is ready for experiment design


In [3]:
# Cell 3 — Controlled stratified 70/15/15 split

def controlled_class_split(group, seed):
    group = group.sample(frac=1, random_state=seed).copy()
    count = len(group)
    test_count = max(1, round(count * 0.15))
    validation_count = max(1, round(count * 0.15))
    train_count = count - validation_count - test_count
    group["split"] = ["train"] * train_count + ["validation"] * validation_count + ["test"] * test_count
    group["protocol"] = "controlled"
    group["independent_source_test"] = False
    return group

controlled_parts = []
for index, (_, group) in enumerate(dataset.groupby("flavor_id", sort=True)):
    controlled_parts.append(controlled_class_split(group, SEED + index))

controlled_split = pd.concat(controlled_parts, ignore_index=True)
controlled_path = SPLIT_DIR / "controlled_stratified_split.csv"
controlled_split.to_csv(controlled_path, index=False)

controlled_summary = pd.crosstab(controlled_split["flavor_id"], controlled_split["split"])
display(controlled_summary)

print("Train:", int(controlled_split["split"].eq("train").sum()))
print("Validation:", int(controlled_split["split"].eq("validation").sum()))
print("Test:", int(controlled_split["split"].eq("test").sum()))
print("Total:", len(controlled_split))
print("Saved:", controlled_path)

assert len(controlled_split) == len(dataset)
assert controlled_split["sample_id"].is_unique
assert set(controlled_split["split"]) == {"train", "validation", "test"}
assert controlled_summary.gt(0).all().all()

print("PASS: every class is represented in all controlled splits")

split,test,train,validation
flavor_id,,,
cocoashot_amaretto,2,8,2
cocoashot_bourbon,5,22,5
cocoashot_chocolate_cake,2,12,2
cocoashot_old_fashioned,4,16,4
cocoashot_tequila,7,30,7
cocoashot_whiskey,2,12,2
truffle_amaretto,4,20,4
truffle_bananas_foster,4,22,4
truffle_brownie_batter,4,16,4


Train: 623
Validation: 133
Test: 133
Total: 889
Saved: /homes/mxqasim/Chocolathon/data/experiments/flavor_embeddings_v1/splits/controlled_stratified_split.csv
PASS: every class is represented in all controlled splits


In [4]:
# Cell 4 — Whole-source holdout protocol

source_parts, holdout_records = [], []

for class_index, (flavor_id, group) in enumerate(dataset.groupby("flavor_id", sort=True)):
    sources = sorted(group["source_file"].unique())

    if len(sources) > 1:
        rng = np.random.default_rng(SEED + class_index)
        holdout_source = sources[int(rng.integers(len(sources)))]
        test_group = group[group["source_file"].eq(holdout_source)].copy()
        development_group = group[~group["source_file"].eq(holdout_source)].sample(frac=1, random_state=SEED + class_index).copy()
        validation_count = max(1, round(len(development_group) * 0.20))
        development_group["split"] = ["train"] * (len(development_group) - validation_count) + ["validation"] * validation_count
        test_group["split"] = "test"
        development_group["independent_source_test"] = True
        test_group["independent_source_test"] = True
        source_parts.extend([development_group, test_group])
        holdout_records.append({
            "flavor_id": flavor_id,
            "flavor_name": group["flavor_name"].iloc[0],
            "training_sources": ", ".join(sorted(development_group["source_file"].unique())),
            "holdout_source": holdout_source,
            "train_samples": int(development_group["split"].eq("train").sum()),
            "validation_samples": int(development_group["split"].eq("validation").sum()),
            "test_samples": len(test_group)
        })
    else:
        development_group = group.sample(frac=1, random_state=SEED + class_index).copy()
        validation_count = max(1, round(len(development_group) * 0.20))
        development_group["split"] = ["train"] * (len(development_group) - validation_count) + ["validation"] * validation_count
        development_group["independent_source_test"] = False
        source_parts.append(development_group)

source_holdout_split = pd.concat(source_parts, ignore_index=True)
source_holdout_split["protocol"] = "source_holdout"
holdout_summary = pd.DataFrame(holdout_records)

source_holdout_path = SPLIT_DIR / "source_holdout_split.csv"
holdout_summary_path = SPLIT_DIR / "source_holdout_summary.csv"
source_holdout_split.to_csv(source_holdout_path, index=False)
holdout_summary.to_csv(holdout_summary_path, index=False)

display(holdout_summary)

print("Train:", int(source_holdout_split["split"].eq("train").sum()))
print("Validation:", int(source_holdout_split["split"].eq("validation").sum()))
print("Independent test:", int(source_holdout_split["split"].eq("test").sum()))
print("Independently testable classes:", len(holdout_summary))
print("Saved:", source_holdout_path)

,flavor_id,flavor_name,training_sources,holdout_source,train_samples,validation_samples,test_samples
0,cocoashot_bourbon,Bourbon,IMG_1979.jpg,IMG_1984.jpg,16,4,12
1,cocoashot_tequila,Tequila,IMG_1980.jpg,IMG_1981.jpg,18,4,22
2,truffle_creamsicle,Creamsicle,IMG_1996.jpg,IMG_1989.jpg,22,6,27
3,truffle_s_mores,S'mores,IMG_1971.jpg,IMG_1985.jpg,19,5,28


Train: 637
Validation: 163
Independent test: 89
Independently testable classes: 4
Saved: /homes/mxqasim/Chocolathon/data/experiments/flavor_embeddings_v1/splits/source_holdout_split.csv


In [5]:
# Cell 5 — Leakage checks and experiment metadata

def split_overlap(frame, first, second):
    first_ids = set(frame.loc[frame["split"].eq(first), "sample_id"])
    second_ids = set(frame.loc[frame["split"].eq(second), "sample_id"])
    return len(first_ids & second_ids)

controlled_overlaps = {
    "train_validation": split_overlap(controlled_split, "train", "validation"),
    "train_test": split_overlap(controlled_split, "train", "test"),
    "validation_test": split_overlap(controlled_split, "validation", "test")
}

source_leakage = []
for row in holdout_summary.itertuples(index=False):
    class_rows = source_holdout_split[source_holdout_split["flavor_id"].eq(row.flavor_id)]
    development_sources = set(class_rows.loc[class_rows["split"].isin(["train", "validation"]), "source_file"])
    test_sources = set(class_rows.loc[class_rows["split"].eq("test"), "source_file"])
    if development_sources & test_sources:
        source_leakage.append(row.flavor_id)

split_metadata = {
    "seed": SEED,
    "dataset_version": "chocolate_dataset_v1",
    "samples": len(dataset),
    "classes": int(dataset["flavor_id"].nunique()),
    "controlled_split": {"ratios": {"train": 0.70, "validation": 0.15, "test": 0.15}, "independent_source_test": False},
    "source_holdout": {
        "independently_testable_classes": holdout_summary["flavor_id"].tolist(),
        "independent_test_samples": int(source_holdout_split["split"].eq("test").sum())
    }
}

metadata_path = SPLIT_DIR / "split_metadata.json"
metadata_path.write_text(json.dumps(split_metadata, indent=2), encoding="utf-8")

assert all(value == 0 for value in controlled_overlaps.values())
assert not source_leakage
assert controlled_split["sample_id"].nunique() == len(dataset)
assert source_holdout_split["sample_id"].nunique() == len(dataset)

print("Controlled sample overlaps:", controlled_overlaps)
print("Source-holdout leakage:", source_leakage)
print("Controlled classes:", controlled_split["flavor_id"].nunique())
print("Source-independent test classes:", len(holdout_summary))
print("Metadata:", metadata_path)
print("\nPASS: both protocols are complete and leakage checks passed")

Controlled sample overlaps: {'train_validation': 0, 'train_test': 0, 'validation_test': 0}
Source-holdout leakage: []
Controlled classes: 33
Source-independent test classes: 4
Metadata: /homes/mxqasim/Chocolathon/data/experiments/flavor_embeddings_v1/splits/split_metadata.json

PASS: both protocols are complete and leakage checks passed


In [6]:
# Cell 4 — Whole-tray holdout split

source_parts, holdout_records = [], []

for class_index, (flavor_id, group) in enumerate(dataset.groupby("flavor_id", sort=True)):
    sources = sorted(group["source_file"].unique())
    shuffled_seed = SEED + class_index

    if len(sources) > 1:
        rng = np.random.default_rng(shuffled_seed)
        holdout_source = sources[int(rng.integers(len(sources)))]
        test_group = group[group["source_file"].eq(holdout_source)].copy()
        development_group = group[~group["source_file"].eq(holdout_source)].sample(frac=1, random_state=shuffled_seed).copy()
        validation_count = max(1, round(len(development_group) * 0.20))

        development_group["split"] = ["train"] * (len(development_group) - validation_count) + ["validation"] * validation_count
        test_group["split"] = "test"
        development_group["independent_source_test"] = True
        test_group["independent_source_test"] = True
        source_parts.extend([development_group, test_group])

        holdout_records.append({
            "flavor_id": flavor_id,
            "flavor_name": group["flavor_name"].iloc[0],
            "training_sources": ", ".join(sorted(development_group["source_file"].unique())),
            "holdout_source": holdout_source,
            "train_samples": int(development_group["split"].eq("train").sum()),
            "validation_samples": int(development_group["split"].eq("validation").sum()),
            "test_samples": len(test_group)
        })
    else:
        development_group = group.sample(frac=1, random_state=shuffled_seed).copy()
        validation_count = max(1, round(len(development_group) * 0.20))
        development_group["split"] = ["train"] * (len(development_group) - validation_count) + ["validation"] * validation_count
        development_group["independent_source_test"] = False
        source_parts.append(development_group)

source_holdout_split = pd.concat(source_parts, ignore_index=True)
source_holdout_split["protocol"] = "source_holdout"
holdout_summary = pd.DataFrame(holdout_records)

source_holdout_path = SPLIT_DIR / "source_holdout_split.csv"
holdout_summary_path = SPLIT_DIR / "source_holdout_summary.csv"
source_holdout_split.to_csv(source_holdout_path, index=False)
holdout_summary.to_csv(holdout_summary_path, index=False)

display(holdout_summary)
print("Train:", int(source_holdout_split["split"].eq("train").sum()))
print("Validation:", int(source_holdout_split["split"].eq("validation").sum()))
print("Independent test:", int(source_holdout_split["split"].eq("test").sum()))
print("Independently testable classes:", len(holdout_summary))
print("Saved:", source_holdout_path)

,flavor_id,flavor_name,training_sources,holdout_source,train_samples,validation_samples,test_samples
0,cocoashot_bourbon,Bourbon,IMG_1979.jpg,IMG_1984.jpg,16,4,12
1,cocoashot_tequila,Tequila,IMG_1980.jpg,IMG_1981.jpg,18,4,22
2,truffle_creamsicle,Creamsicle,IMG_1996.jpg,IMG_1989.jpg,22,6,27
3,truffle_s_mores,S'mores,IMG_1971.jpg,IMG_1985.jpg,19,5,28


Train: 637
Validation: 163
Independent test: 89
Independently testable classes: 4
Saved: /homes/mxqasim/Chocolathon/data/experiments/flavor_embeddings_v1/splits/source_holdout_split.csv


In [7]:
# Cell 5 — Verify split isolation and save metadata

def overlap_count(frame, split_a, split_b):
    ids_a = set(frame.loc[frame["split"].eq(split_a), "sample_id"])
    ids_b = set(frame.loc[frame["split"].eq(split_b), "sample_id"])
    return len(ids_a & ids_b)

controlled_overlaps = {
    "train_validation": overlap_count(controlled_split, "train", "validation"),
    "train_test": overlap_count(controlled_split, "train", "test"),
    "validation_test": overlap_count(controlled_split, "validation", "test")
}

source_leakage = []
for row in holdout_summary.itertuples(index=False):
    class_rows = source_holdout_split[source_holdout_split["flavor_id"].eq(row.flavor_id)]
    development_sources = set(class_rows.loc[class_rows["split"].isin(["train", "validation"]), "source_file"])
    test_sources = set(class_rows.loc[class_rows["split"].eq("test"), "source_file"])
    if development_sources & test_sources:
        source_leakage.append(row.flavor_id)

split_metadata = {
    "seed": SEED,
    "dataset_version": "chocolate_dataset_v1",
    "samples": len(dataset),
    "classes": int(dataset["flavor_id"].nunique()),
    "controlled_counts": controlled_split["split"].value_counts().to_dict(),
    "source_holdout_counts": source_holdout_split["split"].value_counts().to_dict(),
    "independently_testable_classes": holdout_summary["flavor_id"].tolist()
}

metadata_path = SPLIT_DIR / "split_metadata.json"
metadata_path.write_text(json.dumps(split_metadata, indent=2), encoding="utf-8")

assert all(count == 0 for count in controlled_overlaps.values())
assert not source_leakage
assert controlled_split["sample_id"].nunique() == 889
assert source_holdout_split["sample_id"].nunique() == 889
assert len(holdout_summary) == 4

print("Controlled sample overlaps:", controlled_overlaps)
print("Source-holdout leakage:", source_leakage)
print("Controlled classes:", controlled_split["flavor_id"].nunique())
print("Source-independent test classes:", len(holdout_summary))
print("Metadata:", metadata_path)
print("\nPASS: both protocols are complete and leakage checks passed")

Controlled sample overlaps: {'train_validation': 0, 'train_test': 0, 'validation_test': 0}
Source-holdout leakage: []
Controlled classes: 33
Source-independent test classes: 4
Metadata: /homes/mxqasim/Chocolathon/data/experiments/flavor_embeddings_v1/splits/split_metadata.json

PASS: both protocols are complete and leakage checks passed


In [8]:
%pip install --index-url https://download.pytorch.org/whl/cpu torch==2.7.1 torchvision==0.22.1

Looking in indexes: https://download.pytorch.org/whl/cpu
Note: you may need to restart the kernel to use updated packages.


In [9]:
# Cell 6 — PyTorch environment and device check

import sys
import torch
import torchvision
import torch.nn.functional as F
from torch import nn
from torchvision.models import EfficientNet_B0_Weights, efficientnet_b0

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")
else:
    print("CUDA is unavailable; embedding extraction will run on CPU.")

Python: 3.9.25
PyTorch: 2.7.1+cpu
Torchvision: 0.22.1+cpu
Device: cpu
CUDA is unavailable; embedding extraction will run on CPU.


/homes/mxqasim/Chocolathon/.venv/lib64/python3.9/site-packages/networkx/utils/backends.py:135: RuntimeWarning: networkx backend defined more than once: nx-loopback
  backends.update(_get_backends("networkx.backends"))


In [11]:
# Cell 7 — Load frozen EfficientNet-B0 and test one embedding

weights = EfficientNet_B0_Weights.DEFAULT
embedding_transform = weights.transforms()

embedding_model = efficientnet_b0(weights=weights)
embedding_dimension = embedding_model.classifier[1].in_features
embedding_model.classifier = nn.Identity()
embedding_model = embedding_model.to(DEVICE).eval()

for parameter in embedding_model.parameters():
    parameter.requires_grad = False

test_row = dataset.iloc[0]
with Image.open(test_row["image_path"]) as image:
    test_tensor = embedding_transform(ImageOps.exif_transpose(image).convert("RGB")).unsqueeze(0).to(DEVICE)

with torch.inference_mode():
    test_embedding = F.normalize(embedding_model(test_tensor), dim=1)

print("Backbone: EfficientNet-B0")
print("Pretrained weights:", weights)
print("Trainable parameters:", sum(parameter.numel() for parameter in embedding_model.parameters() if parameter.requires_grad))
print("Embedding shape:", tuple(test_embedding.shape))
print("Embedding dimension:", embedding_dimension)
print("Embedding L2 norm:", round(float(test_embedding.norm().cpu()), 6))

assert test_embedding.shape == (1, 1280)
assert torch.isfinite(test_embedding).all()
assert abs(float(test_embedding.norm().cpu()) - 1.0) < 1e-5

print("PASS: frozen embedding backbone is ready")

Backbone: EfficientNet-B0
Pretrained weights: EfficientNet_B0_Weights.IMAGENET1K_V1
Trainable parameters: 0
Embedding shape: (1, 1280)
Embedding dimension: 1280
Embedding L2 norm: 1.0
PASS: frozen embedding backbone is ready


In [11]:
# Cell 8 — Dataset loader for deterministic embedding extraction

import os
from torch.utils.data import Dataset, DataLoader

class ChocolateEmbeddingDataset(Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(row["image_path"]) as image:
            tensor = self.transform(ImageOps.exif_transpose(image).convert("RGB"))
        return tensor, index

torch.set_num_threads(min(8, os.cpu_count() or 1))

embedding_dataset = ChocolateEmbeddingDataset(dataset, embedding_transform)
embedding_loader = DataLoader(
    embedding_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

test_batch, test_indices = next(iter(embedding_loader))

print("Dataset images:", len(embedding_dataset))
print("Batches:", len(embedding_loader))
print("Batch shape:", tuple(test_batch.shape))
print("Batch indices:", test_indices[:5].tolist())
print("CPU threads:", torch.get_num_threads())

assert test_batch.shape[1:] == (3, 224, 224)
assert test_indices[:5].tolist() == [0, 1, 2, 3, 4]

print("PASS: deterministic embedding loader is ready")

Dataset images: 889
Batches: 28
Batch shape: (32, 3, 224, 224)
Batch indices: [0, 1, 2, 3, 4]
CPU threads: 8
PASS: deterministic embedding loader is ready


In [12]:
# Cell 9 — Extract, validate, and cache all embeddings

all_embeddings, all_indices = [], []

with torch.inference_mode():
    for batch_number, (images, indices) in enumerate(embedding_loader, start=1):
        embeddings = F.normalize(embedding_model(images.to(DEVICE)), dim=1)
        all_embeddings.append(embeddings.cpu().numpy().astype(np.float32))
        all_indices.extend(indices.tolist())

        if batch_number == 1 or batch_number % 5 == 0 or batch_number == len(embedding_loader):
            print(f"Processed batch {batch_number}/{len(embedding_loader)}")

embeddings = np.concatenate(all_embeddings, axis=0)
all_indices = np.asarray(all_indices)
embedding_index = dataset.iloc[all_indices].copy().reset_index(drop=True)
embedding_index.insert(0, "embedding_index", np.arange(len(embedding_index)))

embedding_cache_path = OUTPUT_DIR / "efficientnet_b0_embeddings.npz"
embedding_index_path = OUTPUT_DIR / "embedding_index.csv"

np.savez_compressed(
    embedding_cache_path,
    embeddings=embeddings,
    sample_ids=embedding_index["sample_id"].to_numpy(),
    flavor_ids=embedding_index["flavor_id"].to_numpy(),
    source_files=embedding_index["source_file"].to_numpy()
)
embedding_index.drop(columns=["image_path"], errors="ignore").to_csv(embedding_index_path, index=False)

norms = np.linalg.norm(embeddings, axis=1)

print("\nEmbedding matrix:", embeddings.shape)
print("Data type:", embeddings.dtype)
print("Minimum norm:", round(float(norms.min()), 6))
print("Maximum norm:", round(float(norms.max()), 6))
print("Finite values:", bool(np.isfinite(embeddings).all()))
print("Cache size:", round(embedding_cache_path.stat().st_size / 1024**2, 2), "MB")
print("Embedding cache:", embedding_cache_path)
print("Index manifest:", embedding_index_path)

assert embeddings.shape == (889, 1280)
assert np.isfinite(embeddings).all()
assert np.allclose(norms, 1.0, atol=1e-5)
assert embedding_index["sample_id"].is_unique

print("\nPASS: all frozen chocolate embeddings were extracted and cached")

Processed batch 1/28
Processed batch 5/28
Processed batch 10/28
Processed batch 15/28
Processed batch 20/28
Processed batch 25/28
Processed batch 28/28

Embedding matrix: (889, 1280)
Data type: float32
Minimum norm: 1.0
Maximum norm: 1.0
Finite values: True
Cache size: 4.03 MB
Embedding cache: /homes/mxqasim/Chocolathon/data/experiments/flavor_embeddings_v1/outputs/efficientnet_b0_embeddings.npz
Index manifest: /homes/mxqasim/Chocolathon/data/experiments/flavor_embeddings_v1/outputs/embedding_index.csv

PASS: all frozen chocolate embeddings were extracted and cached


In [13]:
%pip install scikit-learn==1.6.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.5/13.5 MB 42.6 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [scikit-learn] [scikit-learn]
Note: you may need to restart the kernel to use updated packages.


In [14]:
# Recovery cell — Reload cached embeddings and experiment manifests

embedding_cache_path = OUTPUT_DIR / "efficientnet_b0_embeddings.npz"
embedding_index_path = OUTPUT_DIR / "embedding_index.csv"
controlled_path = SPLIT_DIR / "controlled_stratified_split.csv"
source_holdout_path = SPLIT_DIR / "source_holdout_split.csv"

embedding_cache = np.load(embedding_cache_path, allow_pickle=True)
embeddings = embedding_cache["embeddings"].astype(np.float32)
embedding_index = pd.read_csv(embedding_index_path)
controlled_split = pd.read_csv(controlled_path)
source_holdout_split = pd.read_csv(source_holdout_path)

cached_sample_ids = embedding_cache["sample_ids"].astype(str)
index_sample_ids = embedding_index["sample_id"].astype(str).to_numpy()

assert embeddings.shape == (889, 1280)
assert len(embedding_index) == 889
assert np.array_equal(cached_sample_ids, index_sample_ids)
assert controlled_split["sample_id"].nunique() == 889
assert source_holdout_split["sample_id"].nunique() == 889

print("Embeddings:", embeddings.shape)
print("Embedding index:", len(embedding_index))
print("Controlled split:", controlled_split["split"].value_counts().to_dict())
print("Source holdout split:", source_holdout_split["split"].value_counts().to_dict())
print("PASS: cached experiment state restored")

Embeddings: (889, 1280)
Embedding index: 889
Controlled split: {'train': 623, 'validation': 133, 'test': 133}
Source holdout split: {'train': 637, 'validation': 163, 'test': 89}
PASS: cached experiment state restored


In [15]:
# Cell 10 — Build training prototypes and evaluate controlled validation

from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

embedding_lookup = {sample_id: index for index, sample_id in enumerate(embedding_index["sample_id"])}
class_ids = sorted(controlled_split["flavor_id"].unique())
class_to_index = {flavor_id: index for index, flavor_id in enumerate(class_ids)}

def get_embedding_rows(frame):
    indices = [embedding_lookup[sample_id] for sample_id in frame["sample_id"]]
    return embeddings[indices]

train_frame = controlled_split[controlled_split["split"].eq("train")].copy()
validation_frame = controlled_split[controlled_split["split"].eq("validation")].copy()

train_embeddings = get_embedding_rows(train_frame)
validation_embeddings = get_embedding_rows(validation_frame)

prototypes = []
for flavor_id in class_ids:
    class_mask = train_frame["flavor_id"].eq(flavor_id).to_numpy()
    prototype = train_embeddings[class_mask].mean(axis=0)
    prototypes.append(prototype / np.linalg.norm(prototype))

prototypes = np.asarray(prototypes, dtype=np.float32)
validation_scores = validation_embeddings @ prototypes.T
validation_ranking = np.argsort(-validation_scores, axis=1)
validation_predictions = np.asarray(class_ids)[validation_ranking[:, 0]]
validation_top3 = np.asarray(class_ids)[validation_ranking[:, :3]]
validation_truth = validation_frame["flavor_id"].to_numpy()

top1_accuracy = accuracy_score(validation_truth, validation_predictions)
top3_accuracy = np.mean([truth in predictions for truth, predictions in zip(validation_truth, validation_top3)])
macro_f1 = f1_score(validation_truth, validation_predictions, labels=class_ids, average="macro", zero_division=0)

print("Training samples:", len(train_frame))
print("Validation samples:", len(validation_frame))
print("Classes:", len(class_ids))
print("Prototype matrix:", prototypes.shape)
print("Validation top-1 accuracy:", round(top1_accuracy, 4))
print("Validation top-3 accuracy:", round(float(top3_accuracy), 4))
print("Validation macro-F1:", round(macro_f1, 4))

assert prototypes.shape == (33, 1280)
assert validation_scores.shape == (133, 33)
assert np.allclose(np.linalg.norm(prototypes, axis=1), 1.0, atol=1e-5)

Training samples: 623
Validation samples: 133
Classes: 33
Prototype matrix: (33, 1280)
Validation top-1 accuracy: 0.9925
Validation top-3 accuracy: 1.0
Validation macro-F1: 0.9929


In [16]:
# Cell 11 — Save predictions and inspect weakest classes

precision, recall, f1, support = precision_recall_fscore_support(
    validation_truth,
    validation_predictions,
    labels=class_ids,
    zero_division=0
)

class_name_lookup = dataset.drop_duplicates("flavor_id").set_index("flavor_id")["flavor_name"].to_dict()
validation_metrics = pd.DataFrame({
    "flavor_id": class_ids,
    "flavor_name": [class_name_lookup[flavor_id] for flavor_id in class_ids],
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "support": support
}).sort_values(["f1", "recall", "flavor_name"]).reset_index(drop=True)

prediction_rows = []
for row_number, (_, row) in enumerate(validation_frame.reset_index(drop=True).iterrows()):
    ranked_ids = validation_top3[row_number]
    prediction_rows.append({
        "sample_id": row["sample_id"],
        "source_file": row["source_file"],
        "true_flavor_id": row["flavor_id"],
        "true_flavor_name": row["flavor_name"],
        "predicted_flavor_id": validation_predictions[row_number],
        "predicted_flavor_name": class_name_lookup[validation_predictions[row_number]],
        "top1_similarity": float(validation_scores[row_number, validation_ranking[row_number, 0]]),
        "top2_flavor_id": ranked_ids[1],
        "top3_flavor_id": ranked_ids[2],
        "top3_correct": row["flavor_id"] in ranked_ids
    })

validation_predictions_df = pd.DataFrame(prediction_rows)
validation_predictions_path = OUTPUT_DIR / "controlled_validation_prototype_predictions.csv"
validation_metrics_path = OUTPUT_DIR / "controlled_validation_prototype_metrics.csv"
prototype_path = OUTPUT_DIR / "efficientnet_b0_class_prototypes.npz"

validation_predictions_df.to_csv(validation_predictions_path, index=False)
validation_metrics.to_csv(validation_metrics_path, index=False)
np.savez_compressed(prototype_path, prototypes=prototypes, class_ids=np.asarray(class_ids))

display(validation_metrics)
print("\nIncorrect top-1 predictions:", int((validation_predictions_df["true_flavor_id"] != validation_predictions_df["predicted_flavor_id"]).sum()))
print("Recovered within top-3:", int((~validation_predictions_df["true_flavor_id"].eq(validation_predictions_df["predicted_flavor_id"]) & validation_predictions_df["top3_correct"]).sum()))
print("Predictions:", validation_predictions_path)
print("Metrics:", validation_metrics_path)
print("Prototypes:", prototype_path)

,flavor_id,flavor_name,precision,recall,f1,support
0,truffle_dulce_de_leche,Dulce de Leche,1.000000,0.75,0.857143,4
1,truffle_tea_honey,Tea & Honey,0.833333,1.00,0.909091,5
2,cocoashot_amaretto,Amaretto,1.000000,1.00,1.000000,2
3,truffle_amaretto,Amaretto,1.000000,1.00,1.000000,4
4,truffle_bananas_foster,Bananas Foster,1.000000,1.00,1.000000,4
5,cocoashot_bourbon,Bourbon,1.000000,1.00,1.000000,5
6,truffle_brownie_batter,Brownie Batter,1.000000,1.00,1.000000,4
7,truffle_caramel_apple_cider,Caramel Apple Cider,1.000000,1.00,1.000000,4
8,truffle_champagne,Champagne,1.000000,1.00,1.000000,4
9,truffle_cheesecake,Cheesecake,1.000000,1.00,1.000000,4



Incorrect top-1 predictions: 1
Recovered within top-3: 1
Predictions: /homes/mxqasim/Chocolathon/data/experiments/flavor_embeddings_v1/outputs/controlled_validation_prototype_predictions.csv
Metrics: /homes/mxqasim/Chocolathon/data/experiments/flavor_embeddings_v1/outputs/controlled_validation_prototype_metrics.csv
Prototypes: /homes/mxqasim/Chocolathon/data/experiments/flavor_embeddings_v1/outputs/efficientnet_b0_class_prototypes.npz


In [21]:
# Recovery helper for embedding-based evaluations

embedding_lookup = {
    sample_id: index
    for index, sample_id in enumerate(embedding_index["sample_id"].astype(str))
}

def get_embedding_rows(frame):
    sample_ids = frame["sample_id"].astype(str).tolist()
    missing = [sample_id for sample_id in sample_ids if sample_id not in embedding_lookup]
    assert not missing, f"Samples missing from embedding cache: {missing[:5]}"
    return embeddings[[embedding_lookup[sample_id] for sample_id in sample_ids]]

class_ids = sorted(controlled_split["flavor_id"].unique())
class_to_index = {flavor_id: index for index, flavor_id in enumerate(class_ids)}
class_name_lookup = dataset.drop_duplicates("flavor_id").set_index("flavor_id")["flavor_name"].to_dict()

test_rows = get_embedding_rows(source_holdout_split.head(5))

print("Cached samples:", len(embedding_lookup))
print("Classes:", len(class_ids))
print("Recovery test shape:", test_rows.shape)

assert len(embedding_lookup) == 889
assert len(class_ids) == 33
assert test_rows.shape == (5, 1280)

print("PASS: embedding evaluation helpers restored")

NameError: name 'embedding_index' is not defined

In [22]:
# Replacement Cell 12 — Self-contained source-holdout prototype evaluation

from sklearn.metrics import accuracy_score, f1_score

embedding_lookup = {sample_id: index for index, sample_id in enumerate(embedding_index["sample_id"].astype(str))}
class_ids = sorted(controlled_split["flavor_id"].unique())
class_to_index = {flavor_id: index for index, flavor_id in enumerate(class_ids)}
class_name_lookup = dataset.drop_duplicates("flavor_id").set_index("flavor_id")["flavor_name"].to_dict()

source_train = source_holdout_split[source_holdout_split["split"].eq("train")].copy().reset_index(drop=True)
source_test = source_holdout_split[source_holdout_split["split"].eq("test")].copy().reset_index(drop=True)

source_train_indices = [embedding_lookup[sample_id] for sample_id in source_train["sample_id"].astype(str)]
source_test_indices = [embedding_lookup[sample_id] for sample_id in source_test["sample_id"].astype(str)]
source_train_embeddings = embeddings[source_train_indices]
source_test_embeddings = embeddings[source_test_indices]

source_prototypes = []
for flavor_id in class_ids:
    class_mask = source_train["flavor_id"].eq(flavor_id).to_numpy()
    prototype = source_train_embeddings[class_mask].mean(axis=0)
    source_prototypes.append(prototype / np.linalg.norm(prototype))

source_prototypes = np.asarray(source_prototypes, dtype=np.float32)
source_test_scores = source_test_embeddings @ source_prototypes.T
source_test_ranking = np.argsort(-source_test_scores, axis=1)
source_test_predictions = np.asarray(class_ids)[source_test_ranking[:, 0]]
source_test_top3 = np.asarray(class_ids)[source_test_ranking[:, :3]]
source_test_truth = source_test["flavor_id"].to_numpy()

source_top1 = accuracy_score(source_test_truth, source_test_predictions)
source_top3_accuracy = np.mean([truth in predictions for truth, predictions in zip(source_test_truth, source_test_top3)])
tested_class_ids = sorted(source_test["flavor_id"].unique())
source_macro_f1 = f1_score(source_test_truth, source_test_predictions, labels=tested_class_ids, average="macro", zero_division=0)

print("Training samples:", len(source_train))
print("Independent test samples:", len(source_test))
print("Full classifier classes:", len(class_ids))
print("Independently tested classes:", len(tested_class_ids))
print("Whole-tray top-1 accuracy:", round(source_top1, 4))
print("Whole-tray top-3 accuracy:", round(float(source_top3_accuracy), 4))
print("Whole-tray macro-F1:", round(source_macro_f1, 4))

assert source_prototypes.shape == (33, 1280)
assert source_test_scores.shape == (89, 33)
assert source_test_ranking.shape == (89, 33)

print("PASS: source-holdout evaluation completed")

NameError: name 'embedding_index' is not defined

In [17]:
# Cell 13 — Per-class whole-tray results and saved predictions

source_prediction_rows = []

for row_number, (_, row) in enumerate(source_test.reset_index(drop=True).iterrows()):
    ranking = source_test_ranking[row_number]
    ranked_ids = np.asarray(class_ids)[ranking[:3]]
    predicted_id = ranked_ids[0]

    source_prediction_rows.append({
        "sample_id": row["sample_id"],
        "source_file": row["source_file"],
        "true_flavor_id": row["flavor_id"],
        "true_flavor_name": row["flavor_name"],
        "predicted_flavor_id": predicted_id,
        "predicted_flavor_name": class_name_lookup[predicted_id],
        "top1_similarity": float(source_test_scores[row_number, ranking[0]]),
        "correct_similarity": float(source_test_scores[row_number, class_to_index[row["flavor_id"]]]),
        "top2_flavor_id": ranked_ids[1],
        "top3_flavor_id": ranked_ids[2],
        "top1_correct": predicted_id == row["flavor_id"],
        "top3_correct": row["flavor_id"] in ranked_ids
    })

source_predictions_df = pd.DataFrame(source_prediction_rows)

source_class_metrics = (
    source_predictions_df.groupby(["true_flavor_id", "true_flavor_name"], as_index=False)
    .agg(
        samples=("sample_id", "count"),
        top1_correct=("top1_correct", "sum"),
        top3_correct=("top3_correct", "sum"),
        mean_top1_similarity=("top1_similarity", "mean")
    )
)
source_class_metrics["top1_accuracy"] = source_class_metrics["top1_correct"] / source_class_metrics["samples"]
source_class_metrics["top3_accuracy"] = source_class_metrics["top3_correct"] / source_class_metrics["samples"]

source_predictions_path = OUTPUT_DIR / "source_holdout_prototype_predictions.csv"
source_metrics_path = OUTPUT_DIR / "source_holdout_prototype_metrics.csv"
source_prototype_path = OUTPUT_DIR / "source_holdout_class_prototypes.npz"

source_predictions_df.to_csv(source_predictions_path, index=False)
source_class_metrics.to_csv(source_metrics_path, index=False)
np.savez_compressed(source_prototype_path, prototypes=source_prototypes, class_ids=np.asarray(class_ids))

display(source_class_metrics)
print("\nIncorrect top-1:", int((~source_predictions_df["top1_correct"]).sum()))
print("Recovered within top-3:", int((~source_predictions_df["top1_correct"] & source_predictions_df["top3_correct"]).sum()))
print("Predictions:", source_predictions_path)
print("Metrics:", source_metrics_path)

NameError: name 'source_test_ranking' is not defined

In [18]:
required_state = [
    "source_test_ranking",
    "source_test_scores",
    "source_test_predictions",
    "source_test_top3",
    "source_test_truth"
]

missing_state = [name for name in required_state if name not in globals()]

print("Missing variables:", missing_state)
assert not missing_state, "Rerun Cell 12 before Cell 13."
print("PASS: Cell 12 completed; Cell 13 can now run")

Missing variables: ['source_test_ranking', 'source_test_scores', 'source_test_predictions', 'source_test_top3', 'source_test_truth']


AssertionError: Rerun Cell 12 before Cell 13.

In [19]:
# Self-contained whole-tray evaluation — reloads everything from disk

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

search_locations = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
PROJECT_ROOT = next(path for path in search_locations if (path / "data").exists() and (path / "02_chocolate_dataset_creation.ipynb").exists())

BUNDLE_DIR = PROJECT_ROOT / "data" / "chocolates" / "frozen" / "chocolate_dataset_v1"
EXPERIMENT_DIR = PROJECT_ROOT / "data" / "experiments" / "flavor_embeddings_v1"
SPLIT_DIR = EXPERIMENT_DIR / "splits"
OUTPUT_DIR = EXPERIMENT_DIR / "outputs"

dataset = pd.read_csv(BUNDLE_DIR / "metadata" / "accepted_dataset_manifest.csv")
controlled_split = pd.read_csv(SPLIT_DIR / "controlled_stratified_split.csv")
source_holdout_split = pd.read_csv(SPLIT_DIR / "source_holdout_split.csv")
embedding_index = pd.read_csv(OUTPUT_DIR / "embedding_index.csv")
embedding_cache = np.load(OUTPUT_DIR / "efficientnet_b0_embeddings.npz", allow_pickle=True)
embeddings = embedding_cache["embeddings"].astype(np.float32)

embedding_lookup = {sample_id: index for index, sample_id in enumerate(embedding_index["sample_id"].astype(str))}
class_ids = sorted(controlled_split["flavor_id"].unique())
class_to_index = {flavor_id: index for index, flavor_id in enumerate(class_ids)}
class_name_lookup = dataset.drop_duplicates("flavor_id").set_index("flavor_id")["flavor_name"].to_dict()

source_train = source_holdout_split[source_holdout_split["split"].eq("train")].copy().reset_index(drop=True)
source_test = source_holdout_split[source_holdout_split["split"].eq("test")].copy().reset_index(drop=True)

source_train_embeddings = embeddings[[embedding_lookup[sample_id] for sample_id in source_train["sample_id"].astype(str)]]
source_test_embeddings = embeddings[[embedding_lookup[sample_id] for sample_id in source_test["sample_id"].astype(str)]]

source_prototypes = []
for flavor_id in class_ids:
    class_embeddings = source_train_embeddings[source_train["flavor_id"].eq(flavor_id).to_numpy()]
    prototype = class_embeddings.mean(axis=0)
    source_prototypes.append(prototype / np.linalg.norm(prototype))

source_prototypes = np.asarray(source_prototypes, dtype=np.float32)
source_test_scores = source_test_embeddings @ source_prototypes.T
source_test_ranking = np.argsort(-source_test_scores, axis=1)
source_test_predictions = np.asarray(class_ids)[source_test_ranking[:, 0]]
source_test_top3 = np.asarray(class_ids)[source_test_ranking[:, :3]]
source_test_truth = source_test["flavor_id"].to_numpy()
tested_class_ids = sorted(source_test["flavor_id"].unique())

source_top1 = accuracy_score(source_test_truth, source_test_predictions)
source_top3_accuracy = np.mean([truth in predictions for truth, predictions in zip(source_test_truth, source_test_top3)])
source_macro_f1 = f1_score(source_test_truth, source_test_predictions, labels=tested_class_ids, average="macro", zero_division=0)

assert embeddings.shape == (889, 1280)
assert source_prototypes.shape == (33, 1280)
assert source_test_scores.shape == (89, 33)

print("=" * 72)
print("WHOLE-TRAY SOURCE-HOLDOUT EVALUATION")
print("=" * 72)
print("Training samples:", len(source_train))
print("Independent test samples:", len(source_test))
print("Classifier classes:", len(class_ids))
print("Independently tested classes:", len(tested_class_ids))
print("Whole-tray top-1 accuracy:", round(source_top1, 4))
print("Whole-tray top-3 accuracy:", round(float(source_top3_accuracy), 4))
print("Whole-tray macro-F1:", round(source_macro_f1, 4))
print("\nPASS: evaluation completed from saved artifacts")

WHOLE-TRAY SOURCE-HOLDOUT EVALUATION
Training samples: 637
Independent test samples: 89
Classifier classes: 33
Independently tested classes: 4
Whole-tray top-1 accuracy: 0.9551
Whole-tray top-3 accuracy: 1.0
Whole-tray macro-F1: 0.98

PASS: evaluation completed from saved artifacts


In [23]:
# Save source-holdout results and inspect all errors

source_prediction_rows = []

for index, row in source_test.iterrows():
    ranking = source_test_ranking[index]
    ranked_ids = np.asarray(class_ids)[ranking[:3]]
    predicted_id = ranked_ids[0]
    correct_index = class_to_index[row["flavor_id"]]

    source_prediction_rows.append({
        "sample_id": row["sample_id"],
        "source_file": row["source_file"],
        "true_flavor_id": row["flavor_id"],
        "true_flavor_name": row["flavor_name"],
        "predicted_flavor_id": predicted_id,
        "predicted_flavor_name": class_name_lookup[predicted_id],
        "top1_similarity": float(source_test_scores[index, ranking[0]]),
        "correct_similarity": float(source_test_scores[index, correct_index]),
        "similarity_margin": float(source_test_scores[index, ranking[0]] - source_test_scores[index, ranking[1]]),
        "top2_flavor_id": ranked_ids[1],
        "top3_flavor_id": ranked_ids[2],
        "top1_correct": predicted_id == row["flavor_id"],
        "top3_correct": row["flavor_id"] in ranked_ids
    })

source_predictions_df = pd.DataFrame(source_prediction_rows)

source_class_metrics = (
    source_predictions_df.groupby(["true_flavor_id", "true_flavor_name"], as_index=False)
    .agg(
        samples=("sample_id", "count"),
        top1_correct=("top1_correct", "sum"),
        top3_correct=("top3_correct", "sum"),
        mean_top1_similarity=("top1_similarity", "mean"),
        mean_margin=("similarity_margin", "mean")
    )
)
source_class_metrics["top1_accuracy"] = source_class_metrics["top1_correct"] / source_class_metrics["samples"]
source_class_metrics["top3_accuracy"] = source_class_metrics["top3_correct"] / source_class_metrics["samples"]

source_errors = source_predictions_df[~source_predictions_df["top1_correct"]].copy()
error_summary = (
    source_errors.groupby(
        ["true_flavor_name", "predicted_flavor_name"], as_index=False
    ).size().rename(columns={"size": "errors"})
)

source_predictions_path = OUTPUT_DIR / "source_holdout_prototype_predictions.csv"
source_metrics_path = OUTPUT_DIR / "source_holdout_prototype_metrics.csv"
source_errors_path = OUTPUT_DIR / "source_holdout_prototype_errors.csv"
source_prototype_path = OUTPUT_DIR / "source_holdout_class_prototypes.npz"

source_predictions_df.to_csv(source_predictions_path, index=False)
source_class_metrics.to_csv(source_metrics_path, index=False)
source_errors.to_csv(source_errors_path, index=False)
np.savez_compressed(source_prototype_path, prototypes=source_prototypes, class_ids=np.asarray(class_ids))

display(source_class_metrics)
display(error_summary)
display(source_errors[[
    "sample_id", "source_file", "true_flavor_name", "predicted_flavor_name",
    "top1_similarity", "correct_similarity", "top2_flavor_id", "top3_flavor_id"
]])

print("Top-1 errors:", len(source_errors))
print("Top-3 errors:", int((~source_predictions_df["top3_correct"]).sum()))
print("Predictions:", source_predictions_path)
print("Metrics:", source_metrics_path)
print("Errors:", source_errors_path)
print("Prototypes:", source_prototype_path)

NameError: name 'source_test_ranking' is not defined

In [24]:
# Self-contained source-holdout evaluation, saving, and error analysis

from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score, f1_score

PROJECT_ROOT = next(path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (path / "data").exists() and (path / "02_chocolate_dataset_creation.ipynb").exists())
BUNDLE_DIR = PROJECT_ROOT / "data" / "chocolates" / "frozen" / "chocolate_dataset_v1"
EXPERIMENT_DIR = PROJECT_ROOT / "data" / "experiments" / "flavor_embeddings_v1"
SPLIT_DIR, OUTPUT_DIR = EXPERIMENT_DIR / "splits", EXPERIMENT_DIR / "outputs"

dataset = pd.read_csv(BUNDLE_DIR / "metadata" / "accepted_dataset_manifest.csv")
controlled_split = pd.read_csv(SPLIT_DIR / "controlled_stratified_split.csv")
source_holdout_split = pd.read_csv(SPLIT_DIR / "source_holdout_split.csv")
embedding_index = pd.read_csv(OUTPUT_DIR / "embedding_index.csv")
embeddings = np.load(OUTPUT_DIR / "efficientnet_b0_embeddings.npz", allow_pickle=True)["embeddings"].astype(np.float32)

embedding_lookup = {sample_id: index for index, sample_id in enumerate(embedding_index["sample_id"].astype(str))}
class_ids = sorted(controlled_split["flavor_id"].unique())
class_to_index = {flavor_id: index for index, flavor_id in enumerate(class_ids)}
class_name_lookup = dataset.drop_duplicates("flavor_id").set_index("flavor_id")["flavor_name"].to_dict()

source_train = source_holdout_split[source_holdout_split["split"].eq("train")].copy().reset_index(drop=True)
source_test = source_holdout_split[source_holdout_split["split"].eq("test")].copy().reset_index(drop=True)
train_embeddings = embeddings[[embedding_lookup[sample_id] for sample_id in source_train["sample_id"].astype(str)]]
test_embeddings = embeddings[[embedding_lookup[sample_id] for sample_id in source_test["sample_id"].astype(str)]]

prototypes = []
for flavor_id in class_ids:
    class_embeddings = train_embeddings[source_train["flavor_id"].eq(flavor_id).to_numpy()]
    prototype = class_embeddings.mean(axis=0)
    prototypes.append(prototype / np.linalg.norm(prototype))

prototypes = np.asarray(prototypes, dtype=np.float32)
scores = test_embeddings @ prototypes.T
rankings = np.argsort(-scores, axis=1)
predictions = np.asarray(class_ids)[rankings[:, 0]]
top3_predictions = np.asarray(class_ids)[rankings[:, :3]]
truth = source_test["flavor_id"].to_numpy()

prediction_rows = []
for index, row in source_test.iterrows():
    ranked_ids = top3_predictions[index]
    predicted_id = ranked_ids[0]
    prediction_rows.append({
        "sample_id": row["sample_id"],
        "source_file": row["source_file"],
        "true_flavor_id": row["flavor_id"],
        "true_flavor_name": row["flavor_name"],
        "predicted_flavor_id": predicted_id,
        "predicted_flavor_name": class_name_lookup[predicted_id],
        "top1_similarity": float(scores[index, rankings[index, 0]]),
        "correct_similarity": float(scores[index, class_to_index[row["flavor_id"]]]),
        "similarity_margin": float(scores[index, rankings[index, 0]] - scores[index, rankings[index, 1]]),
        "top2_flavor_id": ranked_ids[1],
        "top3_flavor_id": ranked_ids[2],
        "top1_correct": predicted_id == row["flavor_id"],
        "top3_correct": row["flavor_id"] in ranked_ids
    })

predictions_df = pd.DataFrame(prediction_rows)
class_metrics = predictions_df.groupby(["true_flavor_id", "true_flavor_name"], as_index=False).agg(
    samples=("sample_id", "count"),
    top1_correct=("top1_correct", "sum"),
    top3_correct=("top3_correct", "sum"),
    mean_similarity=("top1_similarity", "mean"),
    mean_margin=("similarity_margin", "mean")
)
class_metrics["top1_accuracy"] = class_metrics["top1_correct"] / class_metrics["samples"]
class_metrics["top3_accuracy"] = class_metrics["top3_correct"] / class_metrics["samples"]

errors = predictions_df[~predictions_df["top1_correct"]].copy()
error_summary = errors.groupby(["true_flavor_name", "predicted_flavor_name"], as_index=False).size().rename(columns={"size": "errors"})

predictions_df.to_csv(OUTPUT_DIR / "source_holdout_prototype_predictions.csv", index=False)
class_metrics.to_csv(OUTPUT_DIR / "source_holdout_prototype_metrics.csv", index=False)
errors.to_csv(OUTPUT_DIR / "source_holdout_prototype_errors.csv", index=False)
np.savez_compressed(OUTPUT_DIR / "source_holdout_class_prototypes.npz", prototypes=prototypes, class_ids=np.asarray(class_ids))

top1 = accuracy_score(truth, predictions)
top3 = np.mean([label in candidates for label, candidates in zip(truth, top3_predictions)])
macro_f1 = f1_score(truth, predictions, labels=sorted(source_test["flavor_id"].unique()), average="macro", zero_division=0)

display(class_metrics)
display(error_summary)
display(errors[["sample_id", "source_file", "true_flavor_name", "predicted_flavor_name", "top1_similarity", "correct_similarity", "top2_flavor_id", "top3_flavor_id"]])

print("Top-1 accuracy:", round(top1, 4))
print("Top-3 accuracy:", round(float(top3), 4))
print("Macro-F1:", round(macro_f1, 4))
print("Top-1 errors:", len(errors))
print("Top-3 errors:", int((~predictions_df["top3_correct"]).sum()))
print("PASS: results saved successfully")

,true_flavor_id,true_flavor_name,samples,top1_correct,top3_correct,mean_similarity,mean_margin,top1_accuracy,top3_accuracy
0,cocoashot_bourbon,Bourbon,12,12,12,0.696209,0.170239,1.000000,1.0
1,cocoashot_tequila,Tequila,22,21,22,0.804141,0.056435,0.954545,1.0
2,truffle_creamsicle,Creamsicle,27,27,27,0.747043,0.164750,1.000000,1.0
3,truffle_s_mores,S'mores,28,25,28,0.724373,0.054764,0.892857,1.0


,true_flavor_name,predicted_flavor_name,errors
0,S'mores,Maple Cream,3
1,Tequila,Old Fashioned,1


,sample_id,source_file,true_flavor_name,predicted_flavor_name,top1_similarity,correct_similarity,top2_flavor_id,top3_flavor_id
25,IMG_1981_c014,IMG_1981.jpg,Tequila,Old Fashioned,0.731431,0.721725,cocoashot_tequila,cocoashot_chocolate_cake
82,IMG_1985_c022,IMG_1985.jpg,S'mores,Maple Cream,0.695867,0.638599,truffle_peanut_butter_caramel,truffle_s_mores
87,IMG_1985_c027,IMG_1985.jpg,S'mores,Maple Cream,0.753136,0.746607,truffle_s_mores,truffle_manhattan
88,IMG_1985_c028,IMG_1985.jpg,S'mores,Maple Cream,0.743066,0.697081,truffle_s_mores,truffle_manhattan


Top-1 accuracy: 0.9551
Top-3 accuracy: 1.0
Macro-F1: 0.98
Top-1 errors: 4
Top-3 errors: 0
PASS: results saved successfully


In [25]:
# Prototype versus cosine k-NN comparison

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
from sklearn.neighbors import KNeighborsClassifier

PROJECT_ROOT = next(path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (path / "data").exists() and (path / "02_chocolate_dataset_creation.ipynb").exists())
EXPERIMENT_DIR = PROJECT_ROOT / "data" / "experiments" / "flavor_embeddings_v1"
SPLIT_DIR, OUTPUT_DIR = EXPERIMENT_DIR / "splits", EXPERIMENT_DIR / "outputs"

controlled = pd.read_csv(SPLIT_DIR / "controlled_stratified_split.csv")
source_holdout = pd.read_csv(SPLIT_DIR / "source_holdout_split.csv")
embedding_index = pd.read_csv(OUTPUT_DIR / "embedding_index.csv")
embeddings = np.load(OUTPUT_DIR / "efficientnet_b0_embeddings.npz", allow_pickle=True)["embeddings"].astype(np.float32)
lookup = {sample_id: index for index, sample_id in enumerate(embedding_index["sample_id"].astype(str))}

def matrix(frame):
    return embeddings[[lookup[sample_id] for sample_id in frame["sample_id"].astype(str)]]

controlled_train = controlled[controlled["split"].eq("train")].reset_index(drop=True)
controlled_validation = controlled[controlled["split"].eq("validation")].reset_index(drop=True)
source_train = source_holdout[source_holdout["split"].eq("train")].reset_index(drop=True)
source_test = source_holdout[source_holdout["split"].eq("test")].reset_index(drop=True)

comparison_rows = []
for neighbors in [1, 3, 5, 7]:
    model = KNeighborsClassifier(n_neighbors=neighbors, metric="cosine", weights="distance", algorithm="brute")
    model.fit(matrix(controlled_train), controlled_train["flavor_id"])
    predictions = model.predict(matrix(controlled_validation))
    probabilities = model.predict_proba(matrix(controlled_validation))
    top3 = model.classes_[np.argsort(-probabilities, axis=1)[:, :3]]
    truth = controlled_validation["flavor_id"].to_numpy()

    comparison_rows.append({
        "method": f"cosine_knn_k{neighbors}",
        "neighbors": neighbors,
        "validation_top1": accuracy_score(truth, predictions),
        "validation_top3": np.mean([label in candidates for label, candidates in zip(truth, top3)]),
        "validation_macro_f1": f1_score(truth, predictions, average="macro", zero_division=0)
    })

comparison = pd.DataFrame(comparison_rows).sort_values(["validation_macro_f1", "validation_top1"], ascending=False).reset_index(drop=True)
best_k = int(comparison.iloc[0]["neighbors"])

best_knn = KNeighborsClassifier(n_neighbors=best_k, metric="cosine", weights="distance", algorithm="brute")
best_knn.fit(matrix(source_train), source_train["flavor_id"])
source_predictions = best_knn.predict(matrix(source_test))
source_probabilities = best_knn.predict_proba(matrix(source_test))
source_top3 = best_knn.classes_[np.argsort(-source_probabilities, axis=1)[:, :3]]
source_truth = source_test["flavor_id"].to_numpy()

knn_source_top1 = accuracy_score(source_truth, source_predictions)
knn_source_top3 = np.mean([label in candidates for label, candidates in zip(source_truth, source_top3)])
knn_source_f1 = f1_score(source_truth, source_predictions, labels=sorted(source_test["flavor_id"].unique()), average="macro", zero_division=0)

comparison.to_csv(OUTPUT_DIR / "controlled_validation_knn_comparison.csv", index=False)

display(comparison)
print("Selected k from controlled validation:", best_k)
print("k-NN whole-tray top-1:", round(knn_source_top1, 4))
print("k-NN whole-tray top-3:", round(float(knn_source_top3), 4))
print("k-NN whole-tray macro-F1:", round(knn_source_f1, 4))
print("\nPrototype reference: top-1=0.9551, top-3=1.0000, macro-F1=0.9800")

,method,neighbors,validation_top1,validation_top3,validation_macro_f1
0,cosine_knn_k1,1,0.984962,0.984962,0.985955
1,cosine_knn_k3,3,0.962406,1.000000,0.956142
2,cosine_knn_k5,5,0.962406,1.000000,0.953225
3,cosine_knn_k7,7,0.962406,1.000000,0.936067


Selected k from controlled validation: 1
k-NN whole-tray top-1: 0.8652
k-NN whole-tray top-3: 0.8764
k-NN whole-tray macro-F1: 0.929

Prototype reference: top-1=0.9551, top-3=1.0000, macro-F1=0.9800


In [26]:
# Confidence-margin calibration on controlled validation

from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = next(path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (path / "data").exists() and (path / "02_chocolate_dataset_creation.ipynb").exists())
EXPERIMENT_DIR = PROJECT_ROOT / "data" / "experiments" / "flavor_embeddings_v1"
SPLIT_DIR, OUTPUT_DIR = EXPERIMENT_DIR / "splits", EXPERIMENT_DIR / "outputs"

controlled = pd.read_csv(SPLIT_DIR / "controlled_stratified_split.csv")
embedding_index = pd.read_csv(OUTPUT_DIR / "embedding_index.csv")
embeddings = np.load(OUTPUT_DIR / "efficientnet_b0_embeddings.npz", allow_pickle=True)["embeddings"].astype(np.float32)
lookup = {sample_id: index for index, sample_id in enumerate(embedding_index["sample_id"].astype(str))}
class_ids = sorted(controlled["flavor_id"].unique())

train = controlled[controlled["split"].eq("train")].reset_index(drop=True)
validation = controlled[controlled["split"].eq("validation")].reset_index(drop=True)
train_embeddings = embeddings[[lookup[sample_id] for sample_id in train["sample_id"].astype(str)]]
validation_embeddings = embeddings[[lookup[sample_id] for sample_id in validation["sample_id"].astype(str)]]

prototypes = []
for flavor_id in class_ids:
    class_embeddings = train_embeddings[train["flavor_id"].eq(flavor_id).to_numpy()]
    prototype = class_embeddings.mean(axis=0)
    prototypes.append(prototype / np.linalg.norm(prototype))

prototypes = np.asarray(prototypes)
scores = validation_embeddings @ prototypes.T
rankings = np.argsort(-scores, axis=1)
predictions = np.asarray(class_ids)[rankings[:, 0]]
truth = validation["flavor_id"].to_numpy()
margins = scores[np.arange(len(scores)), rankings[:, 0]] - scores[np.arange(len(scores)), rankings[:, 1]]
correct = predictions == truth

threshold_rows = []
for threshold in [0.00, 0.01, 0.02, 0.03, 0.04, 0.05, 0.075, 0.10, 0.15]:
    confident = margins >= threshold
    threshold_rows.append({
        "margin_threshold": threshold,
        "confident_samples": int(confident.sum()),
        "review_samples": int((~confident).sum()),
        "coverage": float(confident.mean()),
        "confident_accuracy": float(correct[confident].mean()) if confident.any() else np.nan,
        "errors_sent_to_review": int((~correct & ~confident).sum()),
        "errors_left_confident": int((~correct & confident).sum())
    })

threshold_results = pd.DataFrame(threshold_rows)
threshold_results.to_csv(OUTPUT_DIR / "controlled_margin_calibration.csv", index=False)

display(threshold_results)
print("Validation errors:", int((~correct).sum()))
print("Incorrect-sample margins:", margins[~correct].round(6).tolist())
print("Saved:", OUTPUT_DIR / "controlled_margin_calibration.csv")

,margin_threshold,confident_samples,review_samples,coverage,confident_accuracy,errors_sent_to_review,errors_left_confident
0,0.000,133,0,1.000000,0.992481,0,1
1,0.010,129,4,0.969925,1.000000,1,0
2,0.020,124,9,0.932331,1.000000,1,0
3,0.030,121,12,0.909774,1.000000,1,0
4,0.040,119,14,0.894737,1.000000,1,0
5,0.050,115,18,0.864662,1.000000,1,0
6,0.075,96,37,0.721805,1.000000,1,0
7,0.100,75,58,0.563910,1.000000,1,0
8,0.150,39,94,0.293233,1.000000,1,0


Validation errors: 1
Incorrect-sample margins: [0.0013000000035390258]
Saved: /homes/mxqasim/Chocolathon/data/experiments/flavor_embeddings_v1/outputs/controlled_margin_calibration.csv


In [27]:
# Apply validation-selected margin threshold to independent whole-tray results

MARGIN_THRESHOLD = 0.01

source_results_path = OUTPUT_DIR / "source_holdout_prototype_predictions.csv"
source_results = pd.read_csv(source_results_path)

source_results["decision"] = np.where(
    source_results["similarity_margin"] >= MARGIN_THRESHOLD,
    "confident_top1",
    "show_top3"
)

confident = source_results["decision"].eq("confident_top1")
correct = source_results["top1_correct"].astype(bool)

threshold_evaluation = pd.DataFrame([{
    "margin_threshold": MARGIN_THRESHOLD,
    "test_samples": len(source_results),
    "confident_samples": int(confident.sum()),
    "top3_review_samples": int((~confident).sum()),
    "automatic_coverage": float(confident.mean()),
    "confident_top1_accuracy": float(correct[confident].mean()),
    "errors_sent_to_top3": int((~correct & ~confident).sum()),
    "errors_left_confident": int((~correct & confident).sum()),
    "overall_top3_accuracy": float(source_results["top3_correct"].mean())
}])

review_cases = source_results[~confident].copy()
confident_errors = source_results[confident & ~correct].copy()

threshold_evaluation.to_csv(OUTPUT_DIR / "source_holdout_margin_evaluation.csv", index=False)
source_results.to_csv(OUTPUT_DIR / "source_holdout_predictions_with_decision.csv", index=False)

display(threshold_evaluation)
print("\nCases sent to top-3:")
display(review_cases[[
    "sample_id", "true_flavor_name", "predicted_flavor_name",
    "similarity_margin", "top2_flavor_id", "top3_flavor_id"
]])

print("\nErrors that remained confident:")
display(confident_errors[[
    "sample_id", "true_flavor_name", "predicted_flavor_name",
    "similarity_margin", "top2_flavor_id", "top3_flavor_id"
]])

print("Saved:", OUTPUT_DIR / "source_holdout_predictions_with_decision.csv")

,margin_threshold,test_samples,confident_samples,top3_review_samples,automatic_coverage,confident_top1_accuracy,errors_sent_to_top3,errors_left_confident,overall_top3_accuracy
0,0.01,89,87,2,0.977528,0.977011,2,2,1.0



Cases sent to top-3:


,sample_id,true_flavor_name,predicted_flavor_name,similarity_margin,top2_flavor_id,top3_flavor_id
25,IMG_1981_c014,Tequila,Old Fashioned,0.009706,cocoashot_tequila,cocoashot_chocolate_cake
87,IMG_1985_c027,S'mores,Maple Cream,0.006529,truffle_s_mores,truffle_manhattan



Errors that remained confident:


,sample_id,true_flavor_name,predicted_flavor_name,similarity_margin,top2_flavor_id,top3_flavor_id
82,IMG_1985_c022,S'mores,Maple Cream,0.042747,truffle_peanut_butter_caramel,truffle_s_mores
88,IMG_1985_c028,S'mores,Maple Cream,0.045985,truffle_s_mores,truffle_manhattan


Saved: /homes/mxqasim/Chocolathon/data/experiments/flavor_embeddings_v1/outputs/source_holdout_predictions_with_decision.csv


In [28]:
# Build final deployment prototypes from all 889 accepted samples

from pathlib import Path
import json
import numpy as np
import pandas as pd

PROJECT_ROOT = next(path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (path / "data").exists() and (path / "02_chocolate_dataset_creation.ipynb").exists())
BUNDLE_DIR = PROJECT_ROOT / "data" / "chocolates" / "frozen" / "chocolate_dataset_v1"
EXPERIMENT_DIR = PROJECT_ROOT / "data" / "experiments" / "flavor_embeddings_v1"
OUTPUT_DIR = EXPERIMENT_DIR / "outputs"
DEPLOYMENT_DIR = EXPERIMENT_DIR / "deployment"
DEPLOYMENT_DIR.mkdir(parents=True, exist_ok=True)

dataset = pd.read_csv(BUNDLE_DIR / "metadata" / "accepted_dataset_manifest.csv")
embedding_index = pd.read_csv(OUTPUT_DIR / "embedding_index.csv")
embedding_cache = np.load(OUTPUT_DIR / "efficientnet_b0_embeddings.npz", allow_pickle=True)
embeddings = embedding_cache["embeddings"].astype(np.float32)

assert np.array_equal(embedding_cache["sample_ids"].astype(str), embedding_index["sample_id"].astype(str).to_numpy())

class_ids = sorted(dataset["flavor_id"].unique())
deployment_prototypes = []

for flavor_id in class_ids:
    sample_ids = set(dataset.loc[dataset["flavor_id"].eq(flavor_id), "sample_id"].astype(str))
    indices = embedding_index.index[embedding_index["sample_id"].astype(str).isin(sample_ids)].to_numpy()
    prototype = embeddings[indices].mean(axis=0)
    deployment_prototypes.append(prototype / np.linalg.norm(prototype))

deployment_prototypes = np.asarray(deployment_prototypes, dtype=np.float32)
class_catalog = (
    dataset.groupby(["flavor_id", "family", "flavor_name"], as_index=False)
    .agg(samples=("sample_id", "count"), source_trays=("source_file", "nunique"))
    .sort_values("flavor_id")
    .reset_index(drop=True)
)
class_catalog.insert(0, "class_index", np.arange(len(class_catalog)))

np.savez_compressed(
    DEPLOYMENT_DIR / "flavor_prototypes.npz",
    prototypes=deployment_prototypes,
    class_ids=np.asarray(class_ids)
)
class_catalog.to_csv(DEPLOYMENT_DIR / "class_catalog.csv", index=False)

print("Deployment prototypes:", deployment_prototypes.shape)
print("Classes:", len(class_catalog))
print("Samples used:", int(class_catalog["samples"].sum()))
print("Prototype file:", DEPLOYMENT_DIR / "flavor_prototypes.npz")
print("Class catalog:", DEPLOYMENT_DIR / "class_catalog.csv")

assert deployment_prototypes.shape == (33, 1280)
assert np.allclose(np.linalg.norm(deployment_prototypes, axis=1), 1.0, atol=1e-5)
assert class_catalog["samples"].sum() == 889

print("PASS: deployment prototypes created")

Deployment prototypes: (33, 1280)
Classes: 33
Samples used: 889
Prototype file: /homes/mxqasim/Chocolathon/data/experiments/flavor_embeddings_v1/deployment/flavor_prototypes.npz
Class catalog: /homes/mxqasim/Chocolathon/data/experiments/flavor_embeddings_v1/deployment/class_catalog.csv
PASS: deployment prototypes created


In [29]:
# Save deployment configuration and baseline results

deployment_config = {
    "version": "flavor_embeddings_v1",
    "dataset_version": "chocolate_dataset_v1",
    "backbone": "efficientnet_b0",
    "weights": "EfficientNet_B0_Weights.IMAGENET1K_V1",
    "embedding_dimension": 1280,
    "similarity": "cosine",
    "classifier": "class_prototype",
    "margin_threshold": 0.01,
    "uncertain_action": "show_top3",
    "supported_classes": 33,
    "unsupported_catalog_classes": ["Blackberry Sangria", "Mint", "Peach Cobbler"],
    "controlled_validation": {
        "samples": 133,
        "top1_accuracy": 0.9925,
        "top3_accuracy": 1.0,
        "macro_f1": 0.9929
    },
    "source_holdout_evaluation": {
        "samples": 89,
        "classes": 4,
        "top1_accuracy": 0.9551,
        "top3_accuracy": 1.0,
        "macro_f1": 0.98
    },
    "limitations": [
        "Controlled validation shares source photographs across splits.",
        "Independent source evaluation covers only four classes.",
        "Most training crops contain white tray backgrounds.",
        "Predictions for unsupported classes must not be forced into supported labels."
    ]
}

config_path = DEPLOYMENT_DIR / "deployment_config.json"
config_path.write_text(json.dumps(deployment_config, indent=2), encoding="utf-8")

required_files = [
    DEPLOYMENT_DIR / "flavor_prototypes.npz",
    DEPLOYMENT_DIR / "class_catalog.csv",
    DEPLOYMENT_DIR / "deployment_config.json"
]

assert all(path.exists() for path in required_files)

print("Deployment package:")
for path in required_files:
    print("-", path)
print("\nPASS: scalable flavor embedding baseline is frozen")

Deployment package:
- /homes/mxqasim/Chocolathon/data/experiments/flavor_embeddings_v1/deployment/flavor_prototypes.npz
- /homes/mxqasim/Chocolathon/data/experiments/flavor_embeddings_v1/deployment/class_catalog.csv
- /homes/mxqasim/Chocolathon/data/experiments/flavor_embeddings_v1/deployment/deployment_config.json

PASS: scalable flavor embedding baseline is frozen
